In [1]:
Console.WriteLine("hello world from dotnet-script");

hello world from dotnet-script


## demo timeutil v1

In [ ]:
public static class TimeUtil
{
    public static string ToIsoUtc(string input)
    {
        if (string.IsNullOrWhiteSpace(input))
            return null;

        // parse ưu tiên datetimeoffset để giữ timezone
        if (DateTimeOffset.TryParse(input, out var dto))
            return dto.UtcDateTime.ToString("O"); // iso 8601 utc

        throw new FormatException("invalid time format");
    }
}

// test runner
var testCases = new Dictionary<string, string>
{
    { "2026-02-10T10:00:00Z", "2026-02-10T10:00:00.0000000Z" },
    { "2026-02-10T10:00:00+07:00", "2026-02-10T03:00:00.0000000Z" },
    {
        "2026-02-10 10:00:00",
        DateTimeOffset.Parse("2026-02-10 10:00:00").UtcDateTime.ToString("O")
    },
    {
        "02/10/2026 10:00:00 AM",
        DateTimeOffset.Parse("02/10/2026 10:00:00 AM").UtcDateTime.ToString("O")
    },
    { "", null },
};

foreach (var tc in testCases)
{
    var actual = TimeUtil.ToIsoUtc(tc.Key);

    if (actual != tc.Value)
        throw new Exception($"FAIL input={tc.Key} expected={tc.Value} actual={actual}");

    Console.WriteLine($"PASS {tc.Key} -> {actual}");
}


PASS 2026-02-10T10:00:00Z -> 2026-02-10T10:00:00.0000000Z
PASS 2026-02-10T10:00:00+07:00 -> 2026-02-10T03:00:00.0000000Z
PASS 2026-02-10 10:00:00 -> 2026-02-10T03:00:00.0000000Z
PASS 02/10/2026 10:00:00 AM -> 2026-02-10T03:00:00.0000000Z
PASS  -> 


## demo timeutil v2

In [15]:
// time utils version 2 with better error handling and support for more formats
using System.Text.RegularExpressions;
public static class TimeUtil
{
    public static string ToIsoUtc(string input)
    {
        if (string.IsNullOrWhiteSpace(input))
            return null;

        input = input.Trim().ToLowerInvariant();

        // normalize gmt/utc offset
        input = Regex.Replace(input, @"(gmt|utc)\s*([+-]\d{1,2})$", m =>
        {
            var hour = int.Parse(m.Groups[2].Value);
            return $"{hour:+00;-00}:00";
        });

        // unix seconds
        if (Regex.IsMatch(input, @"^\d{10}$"))
        {
            var dto = DateTimeOffset.FromUnixTimeSeconds(long.Parse(input));
            return dto.UtcDateTime.ToString("O");
        }

        // unix milliseconds
        if (Regex.IsMatch(input, @"^\d{13}$"))
        {
            var dto = DateTimeOffset.FromUnixTimeMilliseconds(long.Parse(input));
            return dto.UtcDateTime.ToString("O");
        }

        if (DateTimeOffset.TryParse(input, out var dto2))
            return dto2.UtcDateTime.ToString("O");

        throw new FormatException("invalid time format");
    }
}

// test runner
var testCases = new Dictionary<string, string>
{
    {"2026-02-10T10:00:00Z", "2026-02-10T10:00:00.0000000Z"}, 
    {"2026-02-10T10:00:00+07:00", "2026-02-10T03:00:00.0000000Z"},
    {"2026-02-10 10:00:00", DateTimeOffset.Parse("2026-02-10 10:00:00").UtcDateTime.ToString("O")},
    {"02/10/2026 10:00:00 AM", DateTimeOffset.Parse("02/10/2026 10:00:00 AM").UtcDateTime.ToString("O")},
    // iso fractional seconds
    {"2026-02-10T10:00:00.123Z", "2026-02-10T10:00:00.1230000Z"},
    {"2026-02-10T10:00:00.123+07:00", "2026-02-10T03:00:00.1230000Z"},
    // offset không có dấu :
    {"2026-02-10T10:00:00+0700", "2026-02-10T03:00:00.0000000Z"},
    // utc text offset
    {"2026-02-10T10:00:00 UTC+7", "2026-02-10T03:00:00.0000000Z"},
    // gmt text offset
    {"2026-02-10T10:00:00 GMT+7", "2026-02-10T03:00:00.0000000Z"},
    // rfc1123
    {"Tue, 10 Feb 2026 10:00:00 GMT", "2026-02-10T10:00:00.0000000Z"},
    // unix seconds
    {"1770717600", DateTimeOffset.FromUnixTimeSeconds(1770717600).UtcDateTime.ToString("O")},
    // unix milliseconds
    {"1770717600000", DateTimeOffset.FromUnixTimeMilliseconds(1770717600000).UtcDateTime.ToString("O")},
    { "", null },
};

foreach (var tc in testCases)
{
    var actual = TimeUtil.ToIsoUtc(tc.Key);

    if (actual != tc.Value)
        throw new Exception($"FAIL input={tc.Key} expected={tc.Value} actual={actual}");

    Console.WriteLine($"PASS {tc.Key} -> {actual}");
}

PASS 2026-02-10T10:00:00Z -> 2026-02-10T10:00:00.0000000Z
PASS 2026-02-10T10:00:00+07:00 -> 2026-02-10T03:00:00.0000000Z
PASS 2026-02-10 10:00:00 -> 2026-02-10T03:00:00.0000000Z
PASS 02/10/2026 10:00:00 AM -> 2026-02-10T03:00:00.0000000Z
PASS 2026-02-10T10:00:00.123Z -> 2026-02-10T10:00:00.1230000Z
PASS 2026-02-10T10:00:00.123+07:00 -> 2026-02-10T03:00:00.1230000Z
PASS 2026-02-10T10:00:00+0700 -> 2026-02-10T03:00:00.0000000Z
PASS 2026-02-10T10:00:00 UTC+7 -> 2026-02-10T03:00:00.0000000Z
PASS 2026-02-10T10:00:00 GMT+7 -> 2026-02-10T03:00:00.0000000Z
PASS Tue, 10 Feb 2026 10:00:00 GMT -> 2026-02-10T10:00:00.0000000Z
PASS 1770717600 -> 2026-02-10T10:00:00.0000000Z
PASS 1770717600000 -> 2026-02-10T10:00:00.0000000Z
PASS  -> 


## demo timeutil v2 check one

In [16]:
using System.Text.RegularExpressions;
public static class TimeUtil
{
    public static string ToIsoUtc(string input)
    {
        if (string.IsNullOrWhiteSpace(input))
            return null;

        input = input.Trim().ToLowerInvariant();

        // normalize gmt/utc offset
        input = Regex.Replace(input, @"(gmt|utc)\s*([+-]\d{1,2})$", m =>
        {
            var hour = int.Parse(m.Groups[2].Value);
            return $"{hour:+00;-00}:00";
        });

        // unix seconds
        if (Regex.IsMatch(input, @"^\d{10}$"))
        {
            var dto = DateTimeOffset.FromUnixTimeSeconds(long.Parse(input));
            return dto.UtcDateTime.ToString("O");
        }

        // unix milliseconds
        if (Regex.IsMatch(input, @"^\d{13}$"))
        {
            var dto = DateTimeOffset.FromUnixTimeMilliseconds(long.Parse(input));
            return dto.UtcDateTime.ToString("O");
        }

        if (DateTimeOffset.TryParse(input, out var dto2))
            return dto2.UtcDateTime.ToString("O");

        throw new FormatException("invalid time format");
    }
    public static DateTime? ToUtcDateTime(string input)
    {
        if (string.IsNullOrWhiteSpace(input))
            return null;

        input = input.Trim();

        // normalize gmt/utc offset
        input = Regex.Replace(input,
            @"\s*(gmt|utc)\s*([+-]\d{1,2})$",
            m =>
            {
                var hour = int.Parse(m.Groups[2].Value);
                return $"{hour:+00;-00}:00";
            },
            RegexOptions.IgnoreCase);

        // normalize +0700 -> +07:00
        input = Regex.Replace(input,
            @"([+-]\d{2})(\d{2})$",
            "$1:$2");

        // unix seconds
        if (Regex.IsMatch(input, @"^\d{10}$"))
            return DateTimeOffset
                .FromUnixTimeSeconds(long.Parse(input))
                .UtcDateTime;

        // unix milliseconds
        if (Regex.IsMatch(input, @"^\d{13}$"))
            return DateTimeOffset
                .FromUnixTimeMilliseconds(long.Parse(input))
                .UtcDateTime;

        if (DateTimeOffset.TryParse(input, out var dto))
            return dto.UtcDateTime;

        throw new FormatException($"invalid time format: {input}");
    }
}
var testCase = "Tue, 10 Feb 2026 10:00:00 GMT";
var expected = DateTimeOffset.Parse(testCase);
Console.WriteLine($"EXPECTED {expected}");
var actual = TimeUtil.ToUtcDateTime(testCase);
Console.WriteLine($"PASS {actual}");


EXPECTED 2/10/2026 10:00:00 AM +00:00
PASS 2/10/2026 10:00:00 AM


## demo timeutil v3 show all

In [3]:
using System.Text.RegularExpressions;
public static class TimeUtil
{
    public static DateTime? ToUtcDateTime(string input)
    {
        if (string.IsNullOrWhiteSpace(input))
            return null;

        input = input.Trim();

        // normalize gmt/utc offset
        input = Regex.Replace(input,
            @"\s*(gmt|utc)\s*([+-]\d{1,2})$",
            m =>
            {
                var hour = int.Parse(m.Groups[2].Value);
                return $"{hour:+00;-00}:00";
            },
            RegexOptions.IgnoreCase);

        // normalize +0700 -> +07:00
        input = Regex.Replace(input,
            @"([+-]\d{2})(\d{2})$",
            "$1:$2");

        // unix seconds
        if (Regex.IsMatch(input, @"^\d{10}$"))
            return DateTimeOffset
                .FromUnixTimeSeconds(long.Parse(input))
                .UtcDateTime;

        // unix milliseconds
        if (Regex.IsMatch(input, @"^\d{13}$"))
            return DateTimeOffset
                .FromUnixTimeMilliseconds(long.Parse(input))
                .UtcDateTime;

        if (DateTimeOffset.TryParse(input, out var dto))
            return dto.UtcDateTime;

        throw new FormatException($"invalid time format: {input}");
    }
}
var testInputs = new[]
{
    "2026-02-10T10:00:00Z",
    "2026-02-10T10:00:00+07:00",
    "2026-02-10 10:00:00",
    "02/10/2026 10:00:00 AM",

    // iso fractional seconds
    "2026-02-10T10:00:00.123Z",
    "2026-02-10T10:00:00.123+07:00",

    // offset không có dấu :
    "2026-02-10T10:00:00+0700",

    // utc text offset
    "2026-02-10T10:00:00 UTC+7",

    // gmt text offset
    "2026-02-10T10:00:00 GMT+7",

    // rfc1123
    "Tue, 10 Feb 2026 10:00:00 GMT",

    // unix seconds
    "1770717600",

    // unix milliseconds
    "1770717600000",

    ""
};
foreach (var input in testInputs)
{
    try
    {
        var result = TimeUtil.ToUtcDateTime(input);
        Console.WriteLine($"{input ?? "null"} -> {(result?.ToString("O") ?? "null")}");
    }
    catch (Exception ex)
    {
        Console.WriteLine($"{input} -> error: {ex.Message}");
    }
}


2026-02-10T10:00:00Z -> 2026-02-10T10:00:00.0000000Z
2026-02-10T10:00:00+07:00 -> 2026-02-10T03:00:00.0000000Z
2026-02-10 10:00:00 -> 2026-02-10T03:00:00.0000000Z
02/10/2026 10:00:00 AM -> 2026-02-10T03:00:00.0000000Z
2026-02-10T10:00:00.123Z -> 2026-02-10T10:00:00.1230000Z
2026-02-10T10:00:00.123+07:00 -> 2026-02-10T03:00:00.1230000Z
2026-02-10T10:00:00+0700 -> 2026-02-10T03:00:00.0000000Z
2026-02-10T10:00:00 UTC+7 -> 2026-02-10T03:00:00.0000000Z
2026-02-10T10:00:00 GMT+7 -> 2026-02-10T03:00:00.0000000Z
Tue, 10 Feb 2026 10:00:00 GMT -> 2026-02-10T10:00:00.0000000Z
1770717600 -> 2026-02-10T10:00:00.0000000Z
1770717600000 -> 2026-02-10T10:00:00.0000000Z
 -> null
